In [2]:
# 单元格 1：从本地缓存导入 California Housing 数据集

from pathlib import Path
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np

cache_candidates = [
    Path.cwd() / "sklearn_data",
    Path.cwd() / "notebooks" / "sklearn_data",
    Path(r"D:\Python for Data Analysis\notebooks\sklearn_data"),
]

data_home = next(
    (
        path for path in cache_candidates
        if (path / "cal_housing_py3.pkz").exists()
    ),
    None
)

if data_home is None:
    raise FileNotFoundError(
        "未找到 California Housing 缓存文件 cal_housing_py3.pkz。"
    )

housing = fetch_california_housing(
    data_home=data_home,
    download_if_missing=False
)

X = housing.data
y = housing.target

df = pd.DataFrame(X, columns=housing.feature_names)
df["MedHouseVal"] = y

print("数据形状：", X.shape)
display(df.head())

数据形状： (20640, 8)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
# 单元格 2：划分数据集，固定随机种子以保证结果可复现

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [4]:
# 单元格 3：KNN 回归模型

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(
        n_neighbors=10,
        weights="distance",
        p=2,
        algorithm="auto"
    ))
])

knn_model.fit(X_train, y_train)

y_pred_knn = knn_model.predict(X_test)

print("KNN RMSE:", mean_squared_error(y_test, y_pred_knn) ** 0.5)
print("KNN MAE:", mean_absolute_error(y_test, y_pred_knn))
print("KNN R²:", r2_score(y_test, y_pred_knn))

KNN RMSE: 0.6452412314531308
KNN MAE: 0.4367677275399463
KNN R²: 0.6822852607999947


In [5]:
# 单元格 4：KNN 参数调优，不使用测试集调参

from sklearn.model_selection import GridSearchCV, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

knn_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor())
    ]),
    param_grid={
        "knn__n_neighbors": [5, 10, 15, 20, 30],
        "knn__weights": ["uniform", "distance"],
        "knn__p": [1, 2],
    },
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

knn_search.fit(X_train, y_train)

best_knn = knn_search.best_estimator_
y_pred_knn = best_knn.predict(X_test)

print("最佳参数：", knn_search.best_params_)
print("交叉验证 RMSE：", -knn_search.best_score_)
print("测试集 RMSE：", mean_squared_error(y_test, y_pred_knn) ** 0.5)
print("测试集 R²：", r2_score(y_test, y_pred_knn))

最佳参数： {'knn__n_neighbors': 10, 'knn__p': 1, 'knn__weights': 'distance'}
交叉验证 RMSE： 0.600974685822049
测试集 RMSE： 0.6049814774186727
测试集 R²： 0.7206959093709783


In [6]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_model = KNeighborsRegressor(
    n_neighbors=10,
    weights="distance",
    algorithm="auto",
    p=2
)

knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)

print("MSE of KNN:", mean_squared_error(y_test, y_pred_knn))
print("RMSE of KNN:", mean_squared_error(y_test, y_pred_knn) ** 0.5)
print("MAE of KNN:", mean_absolute_error(y_test, y_pred_knn))
print("R² of KNN:", r2_score(y_test, y_pred_knn))

MSE of KNN: 0.4163362467671527
RMSE of KNN: 0.6452412314531308
MAE of KNN: 0.4367677275399463
R² of KNN: 0.6822852607999947


In [7]:
%timeit KNeighborsRegressor(n_neighbors=10, weights="distance", algorithm="ball_tree").fit(X_train_scaled, y_train).predict(X_test_scaled)

%timeit KNeighborsRegressor(n_neighbors=10, weights="distance", algorithm="brute").fit(X_train_scaled, y_train).predict(X_test_scaled)

352 ms ± 29 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
48.1 ms ± 774 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
'''
课堂练习：

• 使用k近邻回归器，根据特征值，预测波士顿房价（实际为加州房价）

• 测试不同参数的效果

• 可视化计算结果
'''

'\n课堂练习：\n\n• 使用k近邻回归器，根据特征值，预测波士顿房价（实际为加州房价）\n\n• 测试不同参数的效果\n\n• 可视化计算结果\n'